# 00.08 — NumPy Vectorization: Batched Linear Prediction

**You should already know:** vectors, dot products, matrices, and basic NumPy arrays (00.01–00.07).  
**By the end, you will be able to:** define vectorization, distinguish `*` from `@`, compute scalar and batched predictions, write an explicit loop, reason about shapes, and verify an independent cross-check.  
**Difficulty:** ★★☆☆☆ · **Time:** about 50–70 minutes

All values are synthetic instructional data. This lesson explains mathematical equivalence and shapes; it makes no timing or performance claim.

## 1. The Problem

For one example, a linear prediction is a weighted sum of its features plus a bias. For several examples, how can we express the same rule clearly without changing its mathematical meaning?

We will start with one row, repeat the rule with a Python loop, and then write the whole batch as `X @ w + b`.

## 2. Toy Example or Dataset

```python
import numpy as np

X = np.array(
    [[1, 2], [3, 0], [2, 4], [5, 1]],
    dtype=np.int64,
)
w = np.array([10, -2], dtype=np.int64)
b = np.int64(3)
```

`X` has four rows (examples) and two columns (features). `w` has one weight per feature, and `b` is one scalar bias:

```text
X.shape == (4, 2)
w.shape == (2,)
b.shape == ()
```

The expected predictions are `[9, 33, 15, 51]`.

In [ ]:
import numpy as np

X = np.array(
    [[1, 2], [3, 0], [2, 4], [5, 1]],
    dtype=np.int64,
)
w = np.array([10, -2], dtype=np.int64)
b = np.int64(3)

print("X =\n", X)
print("w =", w, "b =", b)
print("shapes:", X.shape, w.shape, b.shape)

## 3. Intuition Before Math

For row 0, use one feature contribution at a time:

```text
1(10) + 2(-2) + 3 = 10 - 4 + 3 = 9
```

The same scalar rule applies independently to every row. **Vectorization** here means expressing an operation over whole arrays rather than writing an explicit Python loop over individual elements. It does not, by itself, make a timing promise.

## 4. Mathematical Derivation

For row `i`, the prediction is

```text
x_i1 w_1 + x_i2 w_2 + b.
```

The matrix product `X @ w` computes that two-term dot product for each row. Therefore the batch rule is

```text
y = X @ w + b.
```

The two feature dimensions contract, leaving one result per row:

```text
(X @ w).shape == (4,)
(X @ w + b).shape == (4,)
```

The scalar bias has shape `()` and is added to each element of the prediction vector.

## 5. Manual Walkthrough

| Row | Weighted terms | Prediction |
|---|---|---:|
| 0 | `1(10) + 2(-2) + 3` | `9` |
| 1 | `3(10) + 0(-2) + 3` | `33` |
| 2 | `2(10) + 4(-2) + 3` | `15` |
| 3 | `5(10) + 1(-2) + 3` | `51` |

Elementwise multiplication exposes the feature contributions:

```text
X * w =
[[10, -4],
 [30,  0],
 [20, -8],
 [50, -2]]
```

This has shape `(4, 2)`. It is not directly the prediction vector. Reducing across feature columns gives:

```text
(X * w).sum(axis=1) = [6, 30, 12, 48]
(X * w).sum(axis=1) + b = [9, 33, 15, 51]
```

`*` is elementwise multiplication with broadcasting; `@` is matrix multiplication. Thus `X @ w` performs the row-wise sums directly.

In [ ]:
contributions = X * w
reduced_predictions = contributions.sum(axis=1) + b
vectorized_predictions = X @ w + b

print("X * w =\n", contributions)
print("(X * w).sum(axis=1) + b =", reduced_predictions)
print("X @ w + b =", vectorized_predictions)

## 6. Teaching Implementation

An explicit Python loop makes the repeated scalar rule visible:

```python
loop_predictions = np.empty(X.shape[0], dtype=np.int64)
for i in range(X.shape[0]):
    loop_predictions[i] = X[i, 0] * w[0] + X[i, 1] * w[1] + b
```

The whole-array expression is:

```python
vectorized_predictions = X @ w + b
```

Both describe the same row-wise weighted sum.

In [ ]:
loop_predictions = np.empty(X.shape[0], dtype=np.int64)
for i in range(X.shape[0]):
    loop_predictions[i] = X[i, 0] * w[0] + X[i, 1] * w[1] + b

assert loop_predictions.shape == (4,)
assert vectorized_predictions.shape == (4,)
np.testing.assert_array_equal(loop_predictions, [9, 33, 15, 51])
np.testing.assert_array_equal(loop_predictions, vectorized_predictions)
print("loop and vectorized paths agree:", loop_predictions)

## 7. Verification

The fixed teaching result and its shapes can be asserted directly:

```python
assert X.shape == (4, 2)
assert w.shape == (2,)
assert b.shape == ()
assert contributions.shape == (4, 2)
assert (X @ w + b).shape == (4,)
```

These assertions verify this example. They are not a benchmark or a model evaluation.

In [ ]:
expected = np.array([9, 33, 15, 51], dtype=np.int64)
assert X.dtype == np.int64
assert w.dtype == np.int64
assert b.dtype == np.int64
assert X.shape == (4, 2)
assert w.shape == (2,)
assert b.shape == ()
assert contributions.shape == (4, 2)
assert (X @ w).shape == (4,)
assert (X @ w + b).shape == (4,)
np.testing.assert_array_equal(contributions, [[10, -4], [30, 0], [20, -8], [50, -2]])
np.testing.assert_array_equal(reduced_predictions, expected)
np.testing.assert_array_equal(vectorized_predictions, expected)
print("PASS")

## 8. Independent Cross-Check

This second implementation manually accumulates each row and feature. It intentionally does not call `X @ w`, `np.dot`, or `np.matmul`:

```python
y_cross = np.empty(X.shape[0], dtype=np.int64)

for i in range(X.shape[0]):
    total = b
    for j in range(X.shape[1]):
        total += X[i, j] * w[j]
    y_cross[i] = total
```

Its result must be `[9, 33, 15, 51]`, independently agreeing with the vectorized path.

In [ ]:
y_cross = np.empty(X.shape[0], dtype=np.int64)
for i in range(X.shape[0]):
    total = b
    for j in range(X.shape[1]):
        total += X[i, j] * w[j]
    y_cross[i] = total

np.testing.assert_array_equal(y_cross, expected)
print("independent cross-check:", y_cross)

## 9. Visual Explanation

![Three panels show the scalar, loop, and vectorized paths for the same synthetic batch.](assets/numpy_vectorization_paths.png)

The three panels show equivalent paths. The inset identifies `X * w` as the contribution matrix, not the final prediction vector. The visual is explanatory and makes no timing or performance claim.

## 10. Failure Modes

### `(4,)` versus `(4,1)`

`X @ w` has shape `(4,)`: one-dimensional, with four values. In contrast:

```python
X @ w.reshape(2, 1)
```

has shape `(4, 1)`: two-dimensional, with four rows and one column. The values may look similar, but the array shapes are different.

### Required shape errors

```python
X * w.reshape(2, 1)       # raises ValueError
X * np.array([1, 2, 3])   # raises ValueError
```

These examples demonstrate bounded shape behavior. They are not a general broadcasting tutorial.

### Spot the bug

Why does `X * w + b` not produce one prediction per row? Repair it with either a row reduction or matrix multiplication.

<details>
<summary>Solution</summary>

`X * w` is the `(4,2)` contribution matrix. Use `(X * w).sum(axis=1) + b` or `X @ w + b` to obtain `[9, 33, 15, 51]`.
</details>

## 11. Exercises

1. Using the frozen rule, predict the new synthetic row `[4, 3]`.
2. State the shapes of `X * w`, `(X * w).sum(axis=1)`, `X @ w`, `X @ w + b`, and `X @ w.reshape(2,1)`.
3. Explain why `X * w` is not the final prediction vector and repair it.
4. Explain the valid `(4,1)` result and the two required `ValueError` cases.
5. Write the nested-loop cross-check without `@`, `np.dot`, or `np.matmul`.

<details>
<summary>Solutions</summary>

1. `4(10) + 3(-2) + 3 = 37`.
2. `(4,2)`, `(4,)`, `(4,)`, `(4,)`, and `(4,1)`.
3. Use `(X * w).sum(axis=1) + b` or `X @ w + b`.
4. A two-dimensional right operand yields `(4,1)` under `@`; the two incompatible elementwise shapes raise `ValueError`.
5. Accumulate `total = b` over `j` for each `i`, then assign `y_cross[i]`; the result is `[9, 33, 15, 51]`.
</details>

## 12. References

- NumPy documentation: [Array broadcasting](https://numpy.org/doc/1.26/user/basics.broadcasting.html)
- NumPy documentation: [`numpy.matmul`](https://numpy.org/doc/1.26/reference/generated/numpy.matmul.html)
- NumPy documentation: [`numpy.sum`](https://numpy.org/doc/1.26/reference/generated/numpy.sum.html)
- NumPy documentation: [Array objects and `shape`](https://numpy.org/doc/1.26/reference/arrays.html)

These references support the array, broadcasting, reduction, and matrix-multiplication behaviors demonstrated here.